In [1]:
from datasets import load_dataset

SEED = 42
N_SAMPLES = 200

ds = load_dataset("stanfordnlp/sst2", split="validation")
print("full split size:", len(ds))
print("features:", ds.features)

# fixed, reproducible slice
ds = ds.shuffle(seed=SEED).select(range(N_SAMPLES))
print("sample size:", len(ds))

full split size: 872
features: {'idx': Value('int32'), 'sentence': Value('string'), 'label': ClassLabel(names=['negative', 'positive'])}
sample size: 200


In [2]:
label_names = ds.features["label"].names  # confirm which id means positive
print("label names:", label_names)

for ex in ds.select(range(5)):
    print(ex["label"], "|", ex["sentence"])

label names: ['negative', 'positive']
1 | it gets onto the screen just about as much of the novella as one could reasonably expect , and is engrossing and moving in its own right . 
1 | my big fat greek wedding uses stereotypes in a delightful blend of sweet romance and lovingly dished out humor . 
1 | for the most part , director anne-sophie birot 's first feature is a sensitive , extraordinarily well-acted drama . 
1 | cq 's reflection of artists and the love of cinema-and-self suggests nothing less than a new voice that deserves to be considered as a possible successor to the best european directors . 
1 | charles ' entertaining film chronicles seinfeld 's return to stand-up comedy after the wrap of his legendary sitcom , alongside wannabe comic adams ' attempts to get his shot at the big time . 


In [3]:
print(ds.to_pandas()["label"].value_counts())

label
1    107
0     93
Name: count, dtype: int64


In [4]:
import re
import unicodedata

_WS = re.compile(r"\s+")
_URL = re.compile(r"https?://\S+")
_HTML_TAG = re.compile(r"<[^>]+>")
_ZERO_WIDTH = re.compile(r"[\u200b\u200c\u200d\u200e\u200f\ufeff]")

# Undo SST-2-style tokenization: "does n't" -> "doesn't", "movie ." -> "movie."
_DETOK = [
    (re.compile(r"\s+([.,!?;:])"), r"\1"),
    (re.compile(r"\s+(n't|'s|'re|'ve|'ll|'d|'m)\b"), r"\1"),
    (re.compile(r"-lrb-\s*"), "("),
    (re.compile(r"\s*-rrb-"), ")"),
]


def clean_text(text: str, detokenize: bool = False) -> str:
    """Normalize raw text before it goes into a prompt.

    Steps: Unicode NFKC normalization, remove zero-width characters,
    strip HTML tags and URLs, collapse runs of whitespace, trim the ends.
    If detokenize=True, also rejoin pre-tokenized punctuation and
    contractions (useful for SST-2).

    Does not change case or remove words, so the meaning is preserved.
    """
    text = unicodedata.normalize("NFKC", text)
    text = _ZERO_WIDTH.sub("", text)
    text = _HTML_TAG.sub(" ", text)
    text = _URL.sub(" ", text)
    if detokenize:
        for pattern, repl in _DETOK:
            text = pattern.sub(repl, text)
    return _WS.sub(" ", text).strip()

In [5]:
samples = [
    "  it 's a charming   and often affecting journey .  ",
    "this does n't work at all , -lrb- really -rrb- .",
    "Great movie!!\u200b\n\n  Visit https://example.com <br> now",
    "\tTabs\tand\nnewlines  ",
]
for s in samples:
    print(repr(s), "->", repr(clean_text(s, detokenize=True)))

"  it 's a charming   and often affecting journey .  " -> "it's a charming and often affecting journey."
"this does n't work at all , -lrb- really -rrb- ." -> "this doesn't work at all, (really)."
'Great movie!!\u200b\n\n  Visit https://example.com <br> now' -> 'Great movie!! Visit now'
'\tTabs\tand\nnewlines  ' -> 'Tabs and newlines'


In [6]:
for ex in ds.select(range(5)):
    print(repr(ex["sentence"]))
    print(repr(clean_text(ex["sentence"], detokenize=True)))
    print()

'it gets onto the screen just about as much of the novella as one could reasonably expect , and is engrossing and moving in its own right . '
'it gets onto the screen just about as much of the novella as one could reasonably expect, and is engrossing and moving in its own right.'

'my big fat greek wedding uses stereotypes in a delightful blend of sweet romance and lovingly dished out humor . '
'my big fat greek wedding uses stereotypes in a delightful blend of sweet romance and lovingly dished out humor.'

"for the most part , director anne-sophie birot 's first feature is a sensitive , extraordinarily well-acted drama . "
"for the most part, director anne-sophie birot's first feature is a sensitive, extraordinarily well-acted drama."

"cq 's reflection of artists and the love of cinema-and-self suggests nothing less than a new voice that deserves to be considered as a possible successor to the best european directors . "
"cq's reflection of artists and the love of cinema-and-self sug

In [7]:
# One template per task. Every model gets this exact text as the user message.
PROMPT_SST2 = (
    "Classify the sentiment of the following text. "
    "Answer with exactly one word: positive or negative. "
    "Do not add any explanation.\n\nText: {text}\n\nAnswer:"
)


def build_prompt(template: str, text: str, detokenize: bool = False) -> str:
    """Clean the raw text, then fill it into the prompt template."""
    return template.format(text=clean_text(text, detokenize=detokenize))

In [8]:
ex = ds[0]
prompt = build_prompt(PROMPT_SST2, ex["sentence"], detokenize=True)
print(prompt)
print("\ngold label:", ex["label"], "=", label_names[ex["label"]])

Classify the sentiment of the following text. Answer with exactly one word: positive or negative. Do not add any explanation.

Text: it gets onto the screen just about as much of the novella as one could reasonably expect, and is engrossing and moving in its own right.

Answer:

gold label: 1 = positive


In [9]:
import re, time, json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

PROMPT = (
    "Classify the sentiment of the following text. "
    "Answer with exactly one word: positive or negative. "
    "Do not add any explanation.\n\nText: {text}\n\nAnswer:"
)

POS, NEG = {"positive"}, {"negative"}
NEGATION = {"not", "no", "isn", "never"}

def parse_label(raw: str):
    """Return 1 (positive), 0 (negative), or None (unparseable).
    Looks in the first 6 words; a negated label counts as unparseable."""
    words = re.findall(r"[a-z]+", raw.lower())[:6]
    for i, w in enumerate(words):
        if w in POS or w in NEG:
            if i > 0 and words[i - 1] in NEGATION:
                return None
            return 1 if w in POS else 0
    return None

# Step 1.3 check: one fully formatted prompt
ex = ds[0]
print(PROMPT.format(text=clean_text(ex["sentence"], detokenize=True)))

Classify the sentiment of the following text. Answer with exactly one word: positive or negative. Do not add any explanation.

Text: it gets onto the screen just about as much of the novella as one could reasonably expect, and is engrossing and moving in its own right.

Answer:


In [10]:
HF_ID = "Qwen/Qwen2.5-1.5B-Instruct"

torch.cuda.reset_peak_memory_stats()
tok = AutoTokenizer.from_pretrained(HF_ID)
model = AutoModelForCausalLM.from_pretrained(HF_ID, dtype=torch.float16).to("cuda")
model.eval()

Loading weights: 100%|██████████| 338/338 [00:04<00:00, 80.48it/s]


In [11]:
from tqdm.auto import tqdm

MAX_NEW_TOKENS = 16
rows = []
start = time.time()

for i, ex in enumerate(tqdm(ds, total=len(ds))):
    text = clean_text(ex["sentence"], detokenize=True)
    messages = [{"role": "user", "content": PROMPT.format(text=text)}]
    inputs = tok.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to("cuda")

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)

    raw = tok.decode(
        out[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )
    rows.append({"id": i, "text": text, "gold": ex["label"], "raw": raw, "pred": parse_label(raw)})

elapsed = time.time() - start
print(f"{elapsed/len(rows):.2f}s/example | peak VRAM {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

100%|██████████| 200/200 [00:17<00:00, 11.40it/s]

0.09s/example | peak VRAM 3.11 GB


In [12]:
labels = [r["gold"] for r in rows]
preds  = [r["pred"] for r in rows]

assert len(preds) == len(labels) == len(ds)
print("unparseable:", sum(p is None for p in preds), "/", len(preds))
print("pred counts:", {k: preds.count(k) for k in (0, 1, None)})

# eyeball a few raw outputs, including any unparseable ones
for r in rows[:5] + [r for r in rows if r["pred"] is None][:5]:
    print(r["gold"], r["pred"], repr(r["raw"]))

unparseable: 0 / 200
pred counts: {0: 96, 1: 104, None: 0}
1 1 'positive'
1 1 'positive'
1 1 'positive'
1 1 'positive'
1 1 'positive'


In [13]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

y_true = [r["gold"] for r in rows]
n_unparse = sum(r["pred"] is None for r in rows)
# unparseable counts as wrong (flip the gold label), so errors aren't hidden
y_pred = [r["pred"] if r["pred"] is not None else 1 - r["gold"] for r in rows]

result = {
    "model": HF_ID,
    "task": "sst2_validation",
    "n": len(rows),
    "accuracy": round(accuracy_score(y_true, y_pred), 3),
    "macro_f1": round(f1_score(y_true, y_pred, average="macro"), 3),
    "unparseable_rate": n_unparse / len(rows),
    "latency_s_per_ex": round(elapsed / len(rows), 2),
    "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
}
results_df = pd.DataFrame([result])
print(results_df.to_string(index=False))

# sanity check on the suspicious VRAM/latency numbers (no inference needed)
props = torch.cuda.get_device_properties(0)
print(f"\nGPU: {props.name} | total {props.total_memory/1e9:.2f} GB "
      f"| allocated now {torch.cuda.memory_allocated()/1e9:.2f} GB")

                     model            task   n  accuracy  macro_f1  unparseable_rate  latency_s_per_ex  peak_vram_gb
Qwen/Qwen2.5-1.5B-Instruct sst2_validation 200     0.895     0.895               0.0              0.09          3.11

GPU: NVIDIA GeForce RTX 3050 Laptop GPU | total 4.29 GB | allocated now 3.10 GB
